In [0]:
%pip install catboost lightgbm optuna

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression

import optuna

In [0]:
TARGET = 'Will_Buy_EV'
cat_columns = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
yes_no_columns = ['Home_Charging_Possible', 'Subsidy_Available']
ordinal_columns = ['Range_Anxiety_Level']

In [0]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

df_test['source'] = 'test'
df_train['source'] = 'train'

df = pd.concat(
    [df_train, df_test],
    ignore_index = True
)

df[TARGET] = df[TARGET].str.lower().map({"no": 0, "yes": 1})

# EDA

## Info

In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
df.isna().sum()

In [0]:
df.head()

## Adversarial Validation

See if the distributions of train and test sets match

In [0]:
X_adv = df.drop(columns=[TARGET, 'source', 'id'])
y_adv = df['source'].map({'train': 0, 'test': 1})

for c in cat_columns:
    X_adv[c] = X_adv[c].astype("category")

cv_adv = StratifiedKFold(5, shuffle=True, random_state=42)
oof_adv = cross_val_predict(
    LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1),
    X_adv,
    y_adv,
    cv=cv_adv,
    method="predict_proba"
)[:, 1]
print(roc_auc_score(y_adv, oof_adv))

## EDA on Features

In [0]:
# Income to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Annual_Income_USD',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=18
)

In [0]:
# Daily Commute to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Daily_Commute_km',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=18
)

# Feature Engineering

In [0]:
for c in df[yes_no_columns]:
    df[c] = df[c].str.lower().map({'no': 0, 'yes': 1})


df['Range_Anxiety_Level'] = df['Range_Anxiety_Level'].str.lower().map({'low': 0, 'medium': 1, 'high': 2})

# Baseline model (Logistic Regression)

In [0]:
df.head()

In [0]:
X_log_reg = df[df['source'] == 'train'].drop(columns=[TARGET, 'id']) 
y_log_reg = df[df['source'] == 'train'][TARGET]

In [0]:
log_reg_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
    ],
    remainder="passthrough"
)